In [11]:
import matplotlib as mpl
import xgboost as xgb
from sklearn import datasets
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split

import mlflow
import mlflow.xgboost

mpl.use("Agg")

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("XGBoost Example Comparison")

# prepare train and test data
iris = datasets.load_iris()
X = iris.data
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# enable auto logging
mlflow.xgboost.autolog()

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# hyperparameter search: 4 values each for alpha and lambda (L1_ratio) = 16 combinations
alphas = [0.0001, 0.001, 0.01, 0.1]
lambdas = [0.1, 1.0, 10.0, 100.0]

for alpha in alphas:
    for lambda_val in lambdas:
        with mlflow.start_run(run_name=f"XGBoost alpha={alpha}, lambda={lambda_val}"):
            # train model
            params = {
                "max_depth": 6,
                "objective": "multi:softprob",
                "num_class": 3,
                "learning_rate": 0.1,
                "eval_metric": "mlogloss",
                "colsample_bytree": 1.0,
                "subsample": 1.0,
                "seed": 42,
                "alpha": alpha,
                "lambda": lambda_val,
            }
            model = xgb.train(params, dtrain, evals=[(dtrain, "train"), (dtest, "test")], num_boost_round=10)

            # evaluate model
            y_proba = model.predict(dtest)
            y_pred = y_proba.argmax(axis=1)
            loss = log_loss(y_test, y_proba)
            acc = accuracy_score(y_test, y_pred)

            # log metrics
            mlflow.log_metrics({"log_loss": loss, "accuracy": acc})

            # log model with signature and example input
            signature = mlflow.models.signature.infer_signature(X_test, y_proba)
            mlflow.xgboost.log_model(
                model,
                "model",
                signature=signature,
                input_example=X_test[:1]
            )

2026/06/08 00:27:20 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost Example Comparison' does not exist. Creating a new experiment.


[0]	train-mlogloss:0.96548	test-mlogloss:0.95857
[1]	train-mlogloss:0.85362	test-mlogloss:0.83983
[2]	train-mlogloss:0.75910	test-mlogloss:0.73975
[3]	train-mlogloss:0.67825	test-mlogloss:0.65432
[4]	train-mlogloss:0.60746	test-mlogloss:0.58161
[5]	train-mlogloss:0.54673	test-mlogloss:0.51766
[6]	train-mlogloss:0.49280	test-mlogloss:0.46259
[7]	train-mlogloss:0.44539	test-mlogloss:0.41473
[8]	train-mlogloss:0.40331	test-mlogloss:0.37234
[9]	train-mlogloss:0.36621	test-mlogloss:0.33469


2026/06/08 00:27:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 00:27:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost alpha=0.0001, lambda=0.1 at: http://127.0.0.1:5000/#/experiments/3/runs/6bb6efb0015944cfb0ece1669065686f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
[0]	train-mlogloss:0.97141	test-mlogloss:0.96540
[1]	train-mlogloss:0.86471	test-mlogloss:0.85214
[2]	train-mlogloss:0.77374	test-mlogloss:0.75516
[3]	train-mlogloss:0.69537	test-mlogloss:0.67240
[4]	train-mlogloss:0.62732	test-mlogloss:0.60017
[5]	train-mlogloss:0.56688	test-mlogloss:0.53785
[6]	train-mlogloss:0.51463	test-mlogloss:0.48260
[7]	train-mlogloss:0.46771	test-mlogloss:0.43453
[8]	train-mlogloss:0.42617	test-mlogloss:0.39205
[9]	train-mlogloss:0.38924	test-mlogloss:0.35469


2026/06/08 00:27:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 00:27:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

🏃 View run XGBoost alpha=0.1, lambda=1.0 at: http://127.0.0.1:5000/#/experiments/3/runs/7ad799e57a924c8d8c0e3b4c4a956df7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
[0]	train-mlogloss:1.01051	test-mlogloss:1.00724
[1]	train-mlogloss:0.93206	test-mlogloss:0.92395
[2]	train-mlogloss:0.86184	test-mlogloss:0.84933
[3]	train-mlogloss:0.79883	test-mlogloss:0.78232
[4]	train-mlogloss:0.74224	test-mlogloss:0.72339
[5]	train-mlogloss:0.69112	test-mlogloss:0.66889
[6]	train-mlogloss:0.64468	test-mlogloss:0.61960
[7]	train-mlogloss:0.60297	test-mlogloss:0.57609
[8]	train-mlogloss:0.56480	test-mlogloss:0.53646
[9]	train-mlogloss:0.53007	test-mlogloss:0.49946


2026/06/08 00:28:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 00:28:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost alpha=0.1, lambda=10.0 at: http://127.0.0.1:5000/#/experiments/3/runs/eea36905f4f84ddd84fbc521b77b0c5a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
[0]	train-mlogloss:1.07503	test-mlogloss:1.07638
[1]	train-mlogloss:1.05230	test-mlogloss:1.05291
[2]	train-mlogloss:1.03019	test-mlogloss:1.03008
[3]	train-mlogloss:1.00869	test-mlogloss:1.00785
[4]	train-mlogloss:0.98778	test-mlogloss:0.98623
[5]	train-mlogloss:0.96745	test-mlogloss:0.96520
[6]	train-mlogloss:0.94769	test-mlogloss:0.94473
[7]	train-mlogloss:0.92847	test-mlogloss:0.92483
[8]	train-mlogloss:0.90980	test-mlogloss:0.90547
[9]	train-mlogloss:0.89165	test-mlogloss:0.88664


2026/06/08 00:28:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 00:28:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost alpha=0.1, lambda=100.0 at: http://127.0.0.1:5000/#/experiments/3/runs/11792775d1864d39a1b579d4246f410b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


## How to Assess and Pick the Best Run
I enter the MLflow UI and click on the "XGBoost Example Comparison" experiment. I see a list of runs with their parameters, metrics, and artifacts. I can sort the runs by any column, such as "accuracy" or "rmse", to find the best performing run. I can also click on a run to see more details, such as the parameters used, the metrics logged, and any artifacts saved. I saw thus that the best model was "
XGBoost alpha=0.0001, lambda=0.1"